# SPOTnet — Step 04: Boundary-Aware Loss Training

This notebook adds **boundary-aware loss** to the baseline. The boundary loss penalises errors at specular edges, encouraging sharper mask boundaries.

**Pipeline:**
```
Patch Library → Single-Scale Dataset → UNet → BCE + Dice + Boundary Loss → BL Model
```

**Output:** `runs/detector_boundary_only/best.pt`

## 1. Imports & Config

In [ ]:
import os, glob, random, math
import numpy as np
import cv2
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

PATCH_DIR = 'real_specular_library_v1/patches'
MASK_DIR  = 'real_specular_library_v1/masks'
RUN_DIR   = 'runs/detector_boundary_only'
os.makedirs(RUN_DIR, exist_ok=True)

BASE_CH  = 32
EPOCHS   = 20
BATCH    = 32
LR       = 2e-4
POS_W    = 5.0
IMG_SIZE = 128
BL_W     = 1.5   # boundary loss weight

## 2. Model — UNet

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True),
            nn.Conv2d(cout, cout, 3, 1, 1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True))
    def forward(self, x): return self.net(x)

class UNet(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        self.e1 = ConvBlock(3, base)
        self.e2 = ConvBlock(base, base*2)
        self.e3 = ConvBlock(base*2, base*4)
        self.b  = ConvBlock(base*4, base*4)
        self.d2 = ConvBlock(base*4+base*2, base*2)
        self.d1 = ConvBlock(base*2+base, base)
        self.head = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.e1(x)
        s2 = self.e2(F.avg_pool2d(s1, 2))
        s3 = self.e3(F.avg_pool2d(s2, 2))
        b  = self.b(s3)
        x  = F.interpolate(b,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d2(torch.cat([x, s2], dim=1))
        x  = F.interpolate(x,  scale_factor=2, mode='bilinear', align_corners=False)
        x  = self.d1(torch.cat([x, s1], dim=1))
        return self.head(x)

## 3. Boundary-Aware Loss

In [ ]:
def dice_loss(logits, targets, eps=1e-6):
    p = torch.sigmoid(logits)
    num = 2*(p*targets).sum(dim=(2,3))
    den = (p+targets).sum(dim=(2,3)) + eps
    return (1-num/den).mean()

def get_boundary(mask, kernel_size=3):
    """Extract boundary pixels via morphological dilation - erosion."""
    kernel  = torch.ones(1, 1, kernel_size, kernel_size, device=mask.device) / (kernel_size**2)
    dilated = (F.conv2d(mask, kernel, padding=kernel_size//2) > 0.5).float()
    eroded  = (F.conv2d(mask, kernel, padding=kernel_size//2) > 0.99).float()
    return (dilated - eroded).clamp(0, 1)

def boundary_loss(logits, targets, weight=1.5):
    """Extra penalty at mask boundaries."""
    pred     = torch.sigmoid(logits)
    boundary = get_boundary(targets)
    if boundary.sum() < 1:
        return torch.tensor(0.0, device=logits.device)
    return F.binary_cross_entropy(pred*boundary + 1e-6,
                                  targets*boundary, reduction='mean') * weight

def criterion(logits, targets):
    bce  = F.binary_cross_entropy_with_logits(logits, targets,
               pos_weight=torch.tensor([POS_W], device=logits.device))
    dice = dice_loss(logits, targets)
    bl   = boundary_loss(logits, targets, weight=BL_W)
    return bce + dice + bl

## 4. Dataset & Training

In [ ]:
def to_tensor(img):
    if not img.flags['C_CONTIGUOUS']: img = np.ascontiguousarray(img)
    return torch.from_numpy(img).float().permute(2,0,1) / 255.0

class PatchDS(Dataset):
    def __init__(self, patch_dir, mask_dir, size=128):
        patches = sorted(glob.glob(os.path.join(patch_dir, '*.png')))
        mmap = {os.path.basename(m): m for m in glob.glob(os.path.join(mask_dir, '*.png'))}
        self.pairs = [(p, mmap[os.path.basename(p)]) for p in patches if os.path.basename(p) in mmap]
        self.size = size
        print(f'Pairs: {len(self.pairs)}')
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx):
        p, m = self.pairs[idx]
        img  = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(m, 0)
        img  = cv2.resize(img,  (self.size, self.size))
        mask = cv2.resize(mask, (self.size, self.size), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 127).astype(np.uint8)
        if random.random() < 0.5: img = np.fliplr(img);  mask = np.fliplr(mask)
        if random.random() < 0.5: img = np.flipud(img);  mask = np.flipud(mask)
        if random.random() < 0.5:
            k = random.choice([1,2,3])
            img = np.rot90(img,k); mask = np.rot90(mask,k)
        X = to_tensor(np.ascontiguousarray(img))
        M = torch.from_numpy(np.ascontiguousarray(mask.astype(np.float32)))[None]
        return X, M

ds = PatchDS(PATCH_DIR, MASK_DIR)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, num_workers=4, drop_last=True)

net   = UNet(BASE_CH).to(DEVICE)
opt   = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type=='cuda'))

best_loss = float('inf')
for ep in range(1, EPOCHS+1):
    net.train(); losses = []
    for X, M in tqdm(dl, desc=f'Ep {ep}/{EPOCHS}'):
        X, M = X.to(DEVICE), M.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE.type=='cuda')):
            loss = criterion(net(X), M)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update()
        losses.append(loss.item())
    sched.step()
    avg = np.mean(losses)
    print(f'Ep {ep:03d} | loss: {avg:.4f}')
    if avg < best_loss:
        best_loss = avg
        torch.save({'net': net.state_dict()}, f'{RUN_DIR}/best.pt')
    torch.save({'net': net.state_dict()}, f'{RUN_DIR}/last.pt')

print(f'Saved: {RUN_DIR}/best.pt')